# Phase 4: Modeling & Experiments (CLOSED TRACK)
**CLOSED TRACK: One model per L1. Encoder-only. No LLMs.**

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import torch
import yaml

REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.isdir(os.path.join(REPO_DIR, 'data')):
    REPO_DIR = os.getcwd()
sys.path.insert(0, REPO_DIR)

TRAIN = {l1: os.path.join(REPO_DIR, f'data/train/{l1}/kvl_shared_task_{l1}_train.csv') for l1 in ['es','de','cn']}
DEV = {l1: os.path.join(REPO_DIR, f'data/dev/{l1}/kvl_shared_task_{l1}_dev.csv') for l1 in ['es','de','cn']}
PRED_DIR = os.path.join(REPO_DIR, 'results/predictions')
os.makedirs(PRED_DIR, exist_ok=True)

## Exp 0 — Reproduce Official Baseline

In [ ]:
# Placeholder: use upstream finetune.py or equivalent. Target: es≈1.357, de≈1.328, cn≈1.175
# Save predictions CSV (item_id, prediction) to results/predictions/exp0_es_dev.csv etc.
from scripts.evaluate import CLOSED_BASELINES
print('Exp 0 targets:', CLOSED_BASELINES)

## Exp 0.5 — Target Variable Scaling

In [ ]:
# z-score GLMM_score per L1 on train; train on scaled; inverse-transform for eval
for l1 in ['es','de','cn']:
    tr = pd.read_csv(TRAIN[l1])
    mean_y, std_y = tr['GLMM_score'].mean(), tr['GLMM_score'].std()
    print(f'{l1} train GLMM mean={mean_y:.4f} std={std_y:.4f}')

## Exp 1 — XGBoost Feature-Only

In [ ]:
from features.pipeline import FeaturePipeline
from models.xgboost_baseline import XGBoostBaseline

for l1 in ['es','de','cn']:
    train_df = pd.read_csv(TRAIN[l1])
    dev_df = pd.read_csv(DEV[l1])
    pipe = FeaturePipeline(l1=l1, frozen_features_path=os.path.join(REPO_DIR, 'frozen_features.json'), resources_dir=os.path.join(REPO_DIR, 'resources'))
    model = XGBoostBaseline(l1=l1, feature_pipeline=pipe)
    model.train(train_df)
    model.save_predictions(dev_df, os.path.join(PRED_DIR, f'exp1_{l1}_dev.csv'))
    print(l1, model.evaluate(dev_df))

## Exp 2 — Better Transformer (mdeberta-v3-base)

In [ ]:
# Encoder-only: microsoft/mdeberta-v3-base. Same setup as Exp 0, different architecture.
print('Use HuggingFace Trainer with microsoft/mdeberta-v3-base, MSE loss, per L1.')

## Exp 3 — Structured Input Format

In [ ]:
def structured_input(row):
    return f"English: {row['en_target_word']} | POS: {row['en_target_pos']} | Clue: {row['en_target_clue']} | L1 word: {row['L1_source_word']} | Context: {row['L1_context']}"
dev_df = pd.read_csv(DEV['es'])
print(structured_input(dev_df.iloc[0]))

## Exp 4 — Hybrid Model (MAIN CONTRIBUTION)

In [ ]:
from torch.utils.data import DataLoader
from models.hybrid_transformer import HybridTransformerModel, VocabularyDataset, train_hybrid, predict_hybrid
from transformers import AutoTokenizer

FROZEN_PATH = os.path.join(REPO_DIR, 'frozen_features.json')
encoder_name = 'xlm-roberta-base'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seeds = [10, 42, 123, 456, 789]

for l1 in ['es','de','cn']:
    train_df = pd.read_csv(TRAIN[l1])
    dev_df = pd.read_csv(DEV[l1])
    pipe = FeaturePipeline(l1=l1, frozen_features_path=FROZEN_PATH, resources_dir=os.path.join(REPO_DIR, 'resources'))
    pipe.fit(train_df)
    X_tr = pipe.transform(train_df)
    X_dev = pipe.transform(dev_df)
    feat_cols = [c for c in X_tr.columns if c != 'GLMM_score']
    if not feat_cols:
        feat_cols = list(X_tr.drop(columns=['GLMM_score'], errors='ignore').columns)
    X_tr_f = X_tr[feat_cols].fillna(0).values.astype(np.float32)
    X_dev_f = X_dev[feat_cols].fillna(0).values.astype(np.float32)
    texts_tr = train_df.apply(lambda r: f"{r['en_target_word']} [SEP] {r['en_target_clue']} [SEP] {r['L1_source_word']} {r['L1_context']}", axis=1).tolist()
    texts_dev = dev_df.apply(lambda r: f"{r['en_target_word']} [SEP] {r['en_target_clue']} [SEP] {r['L1_source_word']} {r['L1_context']}", axis=1).tolist()
    tokenizer = AutoTokenizer.from_pretrained(encoder_name)
    train_ds = VocabularyDataset(texts_tr, X_tr_f, train_df['GLMM_score'].values, tokenizer)
    dev_ds = VocabularyDataset(texts_dev, X_dev_f, dev_df['GLMM_score'].values, tokenizer)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    dev_loader = DataLoader(dev_ds, batch_size=32)
    feature_dim = X_tr_f.shape[1]
    for seed in seeds:
        torch.manual_seed(seed)
        model = HybridTransformerModel(encoder_name=encoder_name, feature_dim=feature_dim)
        train_hybrid(model, train_loader, dev_loader, transformer_lr=1e-5, mlp_lr=1e-4, epochs=5, patience=2, device=device)
        preds = predict_hybrid(model, dev_loader, device)
        out = pd.DataFrame({'item_id': dev_df['item_id'], 'prediction': preds})
        out.to_csv(os.path.join(PRED_DIR, f'exp4_hybrid_{l1}_seed{seed}_dev.csv'), index=False)
    print(f'Exp 4 {l1}: 5 seeds saved.')

## Exp 5 — Hyperparameter Tuning

In [ ]:
# Random search, max 15 configs, log everything.
print('Implement random search over lr, batch_size, epochs; log to results.')

## Exp 6 — Ensemble

In [ ]:
# 5-seed average; 0.7*hybrid + 0.3*xgboost. Three submission runs.
for l1 in ['es','de','cn']:
    preds = []
    for seed in [10, 42, 123, 456, 789]:
        p = pd.read_csv(os.path.join(PRED_DIR, f'exp4_hybrid_{l1}_seed{seed}_dev.csv'))
        preds.append(p['prediction'].values)
    avg = np.mean(preds, axis=0)
    dev_df = pd.read_csv(DEV[l1])
    out = pd.DataFrame({'item_id': dev_df['item_id'], 'prediction': avg})
    out.to_csv(os.path.join(PRED_DIR, f'exp6_ensemble_{l1}_dev.csv'), index=False)
    print(f'Exp 6 ensemble {l1} saved.')